In [1]:
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from pydantic import BaseModel, Field


#  初始化大模型对象
model = init_chat_model(
    'deepseek-v4-flash',
    extra_body={'thinking': {'type': 'disabled'}}
)


In [2]:
# 定义状态,记录用户输入,最终结果
class IntentState(MessagesState):
    # 继承后这个类身上有两个属性
    intent: str

# 定义意图识别节点
def intent_node(state: IntentState):
    intent = model.invoke(f'''
    根据用户的请求判断用户意图，可选意图：weather、translate或chat。
    user_query: {state['messages'][-1].content}
    ''')
    return {'intent': intent.content}

# 定义三个子节点
# 获取天气的节点
def weather_node(state: IntentState):
    return {'messages': [AIMessage(content='晴 25度')]}

# 翻译节点
def translate_node(state: IntentState):
    result = model.invoke(f'''
    按照用户要求翻译，只输出翻译结果，不要任何解释。
    user_query:{state['messages'][-1].content}
    ''')
    return {'messages': [result]}

# 聊天节点
def chat_node(state: IntentState):
    result = model.invoke(state['messages'])
    return {'messages': [result]}

def intent_router(state: IntentState) -> Literal['weather', 'translate', 'chat']:
    """
    返回三选一值
    这里是工具节点,意图识别节点将结果吐到这里
    然后将大模型识别到的意图丢入路由判断节点,按照返回名称走到下一个分支节点
    :param state: 大模型吐出需要走的节点
    :return: 直接透传大模型需要走的节点
    """
    return state['intent']

In [3]:
# 创建图
graph_builder = StateGraph(IntentState)

# 添加节点
graph_builder.add_node('intent', intent_node)
graph_builder.add_node('weather', weather_node)
graph_builder.add_node('chat', chat_node)
graph_builder.add_node('translate', translate_node)

# 画图连线
graph_builder.add_edge(START,'intent')
# 条件边识别节点
graph_builder.add_conditional_edges('intent',intent_router)

# 走到三选一节点,然后走到end
graph_builder.add_edge('weather',END)
graph_builder.add_edge('chat',END)
graph_builder.add_edge('translate',END)

# 创建图
graph = graph_builder.compile()

# 执行
for question in ['今天杭州天气如何','草尼玛的英语怎么说','我好累']:
    response = graph.invoke(
        {
            'messages':HumanMessage(content=question)
        }
    )
    print(f"'用户提问:{question}' -> 走的节点={response['intent']} -> 最终节点的执行结果:{response['messages'][-1].content}")

'用户提问:今天杭州天气如何' -> 走的节点=weather -> 最终节点的执行结果:晴 25度
'用户提问:草尼玛的英语怎么说' -> 走的节点=translate -> 最终节点的执行结果:Fuck you
'用户提问:我好累' -> 走的节点=chat -> 最终节点的执行结果:我能感受到你此刻的疲惫，这种累可能来自身体，也可能来自心里。如果可以的话，试着给自己一点空间，哪怕只是几分钟——闭上眼睛，做几次深呼吸，让肩膀放松下来。

疲惫有时是身体在提醒我们：需要慢下来，或者需要被看见。你不需要立刻解决所有问题，此刻允许自己“暂时做不到”，也是一种温柔。

如果愿意，可以和我多说一点——是发生了什么消耗你的事，还是那种说不清道不明的持续沉重？我会在这里安静地听。
